# N-HiTS Training Notebook
**Replaces Prophet — 50x faster, better multi-horizon accuracy**

## Architecture
- 3 hierarchical MLP blocks (coarse/medium/fine scale)
- Each block pools input at a different rate → multi-scale temporal learning
- Residual: each block explains part of the signal, passes remainder to next
- Output: predicted return at each day up to 365 days → all 7 timeframes covered

## Instructions
1. Enable GPU: Settings → Accelerator → **GPU T4 x2** or P100
2. Run All (~15-20 min)
3. Download: `nhits_weights.pth`, `nhits_config.json`
4. Place in `models/pretrained/`
5. Restart Streamlit — N-HiTS auto-activates, replacing Prophet

In [ ]:
!pip install yfinance ta -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import json
import warnings
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import yfinance as yf
import ta
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ─── CONFIG (MUST MATCH models/nhits_model.py) ─────────────────────────────
CONTEXT_LEN    = 120    # 120 trading days of input (~6 months)
MAX_HORIZON    = 365    # predict returns for up to 365 days ahead
N_FEATURES     = None   # set after data loading
POOLING_SIZES  = [5, 2, 1]
D_HIDDEN       = 256
N_LAYERS       = 2
DROPOUT        = 0.1
BATCH_SIZE     = 512
NUM_EPOCHS     = 80
LR             = 5e-4
BUY_THRESH     = 0.05
SELL_THRESH    = -0.05

# Training on multiple horizons simultaneously (multi-task learning)
TRAIN_HORIZONS = [7, 15, 30, 90, 180, 365]

TICKERS = [
    'RELIANCE.NS','TCS.NS','INFY.NS','HDFCBANK.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','SBIN.NS','BAJFINANCE.NS','BHARTIARTL.NS','WIPRO.NS',
    'AXISBANK.NS','LT.NS','MARUTI.NS','SUNPHARMA.NS','M&M.NS',
    'KOTAKBANK.NS','ITC.NS','HCLTECH.NS','ASIANPAINT.NS','TITAN.NS',
    'ULTRACEMCO.NS','BAJAJFINSV.NS','POWERGRID.NS','NTPC.NS','ONGC.NS',
    'DRREDDY.NS','DIVISLAB.NS','CIPLA.NS','TECHM.NS','NESTLEIND.NS',
    'JSWSTEEL.NS','TATASTEEL.NS','HINDALCO.NS','COALINDIA.NS','GRASIM.NS',
    'PIIND.NS','PERSISTENT.NS','COFORGE.NS','MPHASIS.NS','KPITTECH.NS',
    'CHOLAFIN.NS','FEDERALBNK.NS','IDFCFIRSTB.NS','ZYDUSLIFE.NS','ALKEM.NS',
]
print(f'Training on {len(TICKERS)} NSE stocks')

In [ ]:
# ─── DATA PIPELINE ─────────────────────────────────────────────────────────
def fetch(ticker):
    try:
        df = yf.download(ticker, period='10y', interval='1d', auto_adjust=True, progress=False)
        if df is None or df.empty or len(df) < 500:
            return None, None, None
        
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        
        c = pd.Series(df['Close'].values.flatten(), index=df.index, name='Close', dtype=float)
        h = pd.Series(df['High'].values.flatten(), index=df.index, name='High', dtype=float)
        lo = pd.Series(df['Low'].values.flatten(), index=df.index, name='Low', dtype=float)
        v = pd.Series(df['Volume'].values.flatten(), index=df.index, name='Volume', dtype=float)
        
        feat = pd.DataFrame(index=df.index)
        for w in [1, 5, 10, 20, 60]:
            feat[f'ret_{w}d'] = np.log(c / c.shift(w))
        for w in [10, 20, 60]:
            feat[f'vol_{w}d'] = c.pct_change().rolling(w).std()
            
        feat['SMA_20'] = ta.trend.sma_indicator(c, 20)
        feat['SMA_50'] = ta.trend.sma_indicator(c, 50)
        feat['price_vs_sma50']  = (c - feat['SMA_50'])  / feat['SMA_50'].replace(0, np.nan)
        feat['price_vs_sma200'] = (c - ta.trend.sma_indicator(c, 200)) / ta.trend.sma_indicator(c, 200).replace(0, np.nan)
        feat['RSI_14']  = ta.momentum.rsi(c, 14)
        feat['StochRSI']= ta.momentum.stochrsi(c, 14)
        macd = ta.trend.MACD(c)
        feat['MACD'] = macd.macd()
        feat['MACD_Hist'] = macd.macd_diff()
        bb = ta.volatility.BollingerBands(c)
        feat['BB_PctB'] = bb.bollinger_pband()
        feat['ATR_14']  = ta.volatility.average_true_range(h, lo, c, 14)
        feat['ADX_14']  = ta.trend.adx(h, lo, c, 14)
        feat['CCI_20']  = ta.trend.cci(h, lo, c, 20)
        feat['Vol_Ratio']= v / v.rolling(20).mean()
        feat['drawdown_52w'] = (c - c.rolling(252).max()) / c.rolling(252).max().replace(0, np.nan)
        
        feat.replace([np.inf, -np.inf], np.nan, inplace=True)
        feat.ffill(inplace=True)
        feat.fillna(0, inplace=True)
        return feat, c, list(feat.columns)
    except Exception as e:
        print(f'  Failed {ticker}: {e}')
        return None, None, None

all_X, all_close, feature_cols = [], [], None
for t in TICKERS:
    print(f'Fetching {t}...')
    X, close, fcols = fetch(t)
    if X is None:
        continue
    if feature_cols is None:
        feature_cols = fcols
    all_X.append(X)
    all_close.append(close)
print(f'Loaded {len(all_X)} stocks | features: {len(feature_cols) if feature_cols else 0}')
N_FEATURES = len(feature_cols)

In [ ]:
# ─── BUILD MULTI-HORIZON DATASET ───────────────────────────────────────────
def build_samples(X, close, context_len, horizons):
    max_h = max(horizons)
    seqs, targets = [], []
    Xv = X.values.astype(np.float32)
    cv = close.values.astype(np.float32)
    for i in range(context_len, len(Xv) - max_h):
        seq = Xv[i-context_len:i]
        ret = []
        for h in horizons:
            r = float(cv[i+h] / cv[i] - 1.0)
            ret.append(np.clip(r, -0.5, 0.5))
        seqs.append(seq)
        targets.append(ret)
    return np.array(seqs), np.array(targets, dtype=np.float32)

all_seqs, all_tgts = [], []
for X, close in zip(all_X, all_close):
    s, t = build_samples(X, close, CONTEXT_LEN, TRAIN_HORIZONS)
    all_seqs.append(s)
    all_tgts.append(t)
X_all = np.concatenate(all_seqs)
y_all = np.concatenate(all_tgts)
print(f'Total samples: {len(X_all):,}  X:{X_all.shape}  y:{y_all.shape}')

# Scale features
n, cl, nf = X_all.shape
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all.reshape(-1, nf)).reshape(n, cl, nf)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

split = int(0.85 * n)
X_tr, X_val = X_scaled[:split], X_scaled[split:]
y_tr, y_val = y_all[:split],   y_all[split:]
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')

In [ ]:
# ─── N-HiTS ARCHITECTURE (MUST MATCH models/nhits_model.py) ───────────────
class NHiTSBlock(nn.Module):
    def __init__(self, ctx, h_len, nf, pool, dh=D_HIDDEN, nl=N_LAYERS, dr=DROPOUT):
        super().__init__()
        self.pool_size = pool
        pool_len = ctx // pool + (1 if ctx % pool else 0)
        inp_dim = pool_len * nf
        layers = [nn.Linear(inp_dim, dh), nn.ReLU(), nn.Dropout(dr)]
        for _ in range(nl-1):
            layers += [nn.Linear(dh, dh), nn.ReLU(), nn.Dropout(dr)]
        self.mlp = nn.Sequential(*layers)
        self.bc_head = nn.Linear(dh, inp_dim)
        self.fc_head = nn.Linear(dh, h_len)
        self.ap = nn.AdaptiveAvgPool1d(pool_len)
        self._ctx = ctx

    def forward(self, x):
        # x: (B, nf, ctx)
        p = self.ap(x)
        B, F, L = p.shape
        h = self.mlp(p.reshape(B, F*L))
        bc = self.bc_head(h).reshape(B, F, L)
        bc_up = nn.functional.interpolate(bc, size=self._ctx, mode='linear', align_corners=False)
        fc = self.fc_head(h)
        return bc_up, fc

class NHiTSNetwork(nn.Module):
    def __init__(self, ctx, h_len, nf, pools=POOLING_SIZES, dh=D_HIDDEN, nl=N_LAYERS, dr=DROPOUT):
        super().__init__()
        self.blocks = nn.ModuleList([
            NHiTSBlock(ctx, h_len, nf, ps, dh, nl, dr) for ps in pools
        ])
        self.h_len = h_len
    def forward(self, x):
        # x: (B, nf, ctx)
        res = x
        total = torch.zeros(x.shape[0], self.h_len, device=x.device)
        for blk in self.blocks:
            bc, fc = blk(res)
            res = res - bc
            total = total + fc
        return total

H_LEN = MAX_HORIZON
model = NHiTSNetwork(CONTEXT_LEN, H_LEN, N_FEATURES).to(DEVICE)
print(f'N-HiTS parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─── TRAINING ──────────────────────────────────────────────────────────────
Xtr_t = torch.FloatTensor(X_tr).permute(0, 2, 1)
Xvl_t = torch.FloatTensor(X_val).permute(0, 2, 1)

horizon_idx = [h-1 for h in TRAIN_HORIZONS]

def custom_loss(pred, target):
    pred_at_h = pred[:, horizon_idx]
    mse = nn.functional.mse_loss(pred_at_h, target)
    direction_pred   = torch.sign(pred_at_h)
    direction_target = torch.sign(target)
    dir_loss = (1 - (direction_pred == direction_target).float()).mean()
    return mse + 0.1 * dir_loss

train_ds = TensorDataset(Xtr_t, torch.FloatTensor(y_tr))
val_ds   = TensorDataset(Xvl_t, torch.FloatTensor(y_val))
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  pin_memory=True, num_workers=2)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=2)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

best_val, pc = 1e9, 0
print('Training N-HiTS...')
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = custom_loss(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    scheduler.step()
    
    model.eval()
    vloss = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            vloss += custom_loss(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
    vloss /= len(val_dl)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | Val Loss: {vloss:.6f}')
    if vloss < best_val:
        best_val = vloss
        torch.save(model.state_dict(), '/kaggle/working/nhits_weights.pth')
        pc = 0
    else:
        pc += 1
        if pc >= 10:
            print(f'Early stop @ {epoch}')
            break
print(f'Best Val Loss: {best_val:.6f}')

In [ ]:
# ─── DIRECTIONAL ACCURACY REPORT ───────────────────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/nhits_weights.pth', map_location=DEVICE))
model.eval()

all_preds, all_tgts_eval = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        p = model(xb.to(DEVICE))[:, horizon_idx].cpu().numpy()
        all_preds.append(p)
        all_tgts_eval.append(yb.numpy())
preds = np.concatenate(all_preds)
tgts = np.concatenate(all_tgts_eval)

print('Directional Accuracy per horizon:')
for i, h in enumerate(TRAIN_HORIZONS):
    p_dir = (preds[:, i] > 0)
    t_dir = (tgts[:, i] > 0)
    acc = (p_dir == t_dir).mean()
    buy_mask = tgts[:, i] > BUY_THRESH
    buy_acc  = (preds[:, i][buy_mask] > 0).mean() if buy_mask.sum() > 0 else 0
    print(f'  {h:3d}d: Dir Acc={acc:.4f}  Buy Recall={buy_acc:.4f}')

In [ ]:
# ─── SAVE CONFIG ───────────────────────────────────────────────────────────
config = {
    'context_len': CONTEXT_LEN,
    'max_horizon': H_LEN,
    'n_features': N_FEATURES,
    'pooling_sizes': POOLING_SIZES,
    'd_hidden': D_HIDDEN,
    'n_layers': N_LAYERS,
    'dropout': DROPOUT,
    'feature_cols': feature_cols,
    'train_tickers': TICKERS,
    'train_horizons': TRAIN_HORIZONS,
    'best_val_loss': round(float(best_val), 6),
}
with open('/kaggle/working/nhits_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: /kaggle/working/nhits_weights.pth')
print('Saved: /kaggle/working/nhits_config.json')
print('\nCopy both to: models/pretrained/')
print('No feature_scaler.pkl needed — N-HiTS uses raw features!')
print('Restart Streamlit — N-HiTS replaces Prophet instantly!')